In [12]:
from typing import Annotated
from typing_extensions import TypedDict
import base64

from fastapi import FastAPI, UploadFile, File, Form
from pydantic import BaseModel, Field

from langchain_core.messages import SystemMessage, HumanMessage
from langchain_openai import ChatOpenAI

from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.checkpoint.memory import InMemorySaver

SYSTEM_PROMPT = """
You are a helpful AI assistant.

Always respond using the provided schema.

Rules:
- "response" is your normal conversational reply.
- "fruits" contains every fruit detected from the uploaded image or mentioned by the user.
- Each fruit should appear only once with its total count.
- If no fruits are detected or mentioned, return an empty list.
- Fruit names must always be written in English.
"""

class Fruit(BaseModel):
    name: str = Field(description="Fruit name in English")
    count: int = Field(description="Number of this fruit")

class ChatResponse(BaseModel):
    response: str = Field(description="Assistant response")
    fruits: list[Fruit] = Field(
        default_factory=list,
        description="Detected fruits from the image or user message."
    )

class AppState(TypedDict):
    all_messages: Annotated[list, add_messages]

llm = ChatOpenAI(
    base_url="http://10.0.38.50:50015/v1",
    api_key="dummy",
    model="dummy",
    temperature=0.1,
).with_structured_output(ChatResponse)

class Nodes:
    def __init__(self, llm):
        self.llm = llm

    def agent_node(self, state: AppState):
        result: ChatResponse = self.llm.invoke(
            [
                SystemMessage(content=SYSTEM_PROMPT),
                *state["all_messages"],
            ]
        )

        return {
            "all_messages": [
                HumanMessage(content=result.model_dump_json())
            ]
        }

builder = StateGraph(AppState)
builder.add_node("agent", Nodes(llm).agent_node)
builder.add_edge(START, "agent")
builder.add_edge("agent", END)

graph = builder.compile(checkpointer=InMemorySaver())

app = FastAPI()

@app.post("/chat", response_model=ChatResponse)
async def chat(
    username: str = Form(...),
    message: str = Form(...),
    image: UploadFile | None = File(None),
):
    content = []

    if message.strip():
        content.append(
            {
                "type": "text",
                "text": message,
            }
        )

    if image is not None:
        image_bytes = await image.read()
        image_b64 = base64.b64encode(image_bytes).decode()

        content.append(
            {
                "type": "image_url",
                "image_url": {
                    "url": f"data:{image.content_type};base64,{image_b64}"
                },
            }
        )

    state = graph.invoke(
        {
            "all_messages": [
                HumanMessage(content=content)
            ]
        },
        config={
            "configurable": {
                "thread_id": username
            }
        },
    )

    result: ChatResponse = llm.invoke(
        [
            SystemMessage(content=SYSTEM_PROMPT),
            *state["all_messages"],
        ]
    )

    return result

In [13]:
import nest_asyncio
import uvicorn

nest_asyncio.apply()

config = uvicorn.Config(app, host="0.0.0.0", port=12345)
server = uvicorn.Server(config)

await server.serve()

INFO:     Started server process [27604]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:12345 (Press CTRL+C to quit)


INFO:     127.0.0.1:63691 - "GET /docs HTTP/1.1" 200 OK
INFO:     127.0.0.1:63691 - "GET /openapi.json HTTP/1.1" 200 OK
INFO:     127.0.0.1:63692 - "POST /chat HTTP/1.1" 422 Unprocessable Entity
INFO:     127.0.0.1:49528 - "POST /chat HTTP/1.1" 200 OK
INFO:     127.0.0.1:49531 - "POST /chat HTTP/1.1" 200 OK
INFO:     127.0.0.1:49569 - "POST /chat HTTP/1.1" 200 OK
INFO:     127.0.0.1:49572 - "POST /chat HTTP/1.1" 200 OK
INFO:     127.0.0.1:49652 - "POST /chat HTTP/1.1" 200 OK


INFO:     Shutting down
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.
INFO:     Finished server process [27604]
